# Data Cleaning: From KOSIS Raw Data to Tidy Format (Python)


## Introduction

This notebook documents a Python workflow for transforming raw KOSIS responses into tidy tables compatible with `pycensuskr` conventions.


In [ ]:
import pandas as pd
from urllib.parse import urlparse, parse_qs


## API URL inspection


In [ ]:
url_tax_general = "https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=YOUR_KEY&orgId=133&tblId=DT_133N_A3212&itmId=T001&prdSe=Y&format=json"

parsed = urlparse(url_tax_general)
params = parse_qs(parsed.query)
pd.DataFrame({k: [','.join(v)] for k, v in params.items()})


## Example: standardize tax data


In [ ]:
# Assume df_tax is loaded from a KOSIS response
# df_tax = pd.DataFrame(response_json)

def clean_tax(df_tax: pd.DataFrame, sgg_lookup: pd.DataFrame) -> pd.DataFrame:
    out = df_tax.rename(columns={'C1': 'adm2_code', 'DT': 'value'})[['adm2_code', 'value']].copy()
    out['adm2_code'] = pd.to_numeric(out['adm2_code'], errors='coerce').astype('Int64')
    out['value'] = pd.to_numeric(out['value'], errors='coerce')
    out = out.merge(
        sgg_lookup[['sgg_tax_global', 'sido_en', 'sigungu_1_en', 'adm2_code']],
        on='adm2_code',
        how='left'
    )
    return out


## Example: standardize population data


In [ ]:
def clean_population(df_pop: pd.DataFrame, sgg_lookup: pd.DataFrame) -> pd.DataFrame:
    sex_map = {'0': 'total', '1': 'male', '2': 'female', 0: 'total', 1: 'male', 2: 'female'}
    itm_map = {'T00': 'population_total', 'T60': 'population_nonrelative'}

    out = df_pop.copy()
    out['sex'] = out['C2'].map(sex_map)
    out['type'] = out['ITM_ID'].map(itm_map)
    out = out[['C1', 'C1_NM', 'sex', 'type', 'DT']]
    out = out.pivot_table(index=['C1', 'C1_NM'], columns=['type', 'sex'], values='DT', aggfunc='first').reset_index()
    out.columns = ['_'.join([str(x) for x in c if x]).strip('_') for c in out.columns.to_flat_index()]
    out = out.rename(columns={'C1': 'sigungu_cd', 'C1_NM': 'sigungu_kr'})
    out = out.merge(sgg_lookup[['adm2_code', 'sido_en', 'sigungu_1_en']], on='adm2_code', how='left')
    return out


## Long-format tidy schema


In [ ]:
# Example target columns for pycensuskr-compatible long table
target_cols = [
    'year', 'adm1', 'adm2', 'adm2_code',
    'type', 'class1', 'class2', 'unit', 'value'
]
pd.DataFrame({'column': target_cols})
